# Turing Data Analysis MCQ — Consolidated Notebook

All questions and their pandas/scipy answers in one runnable file.

**Sections**
1. Setup (imports + data download + preprocessing)
2. Reference Questions — Cardiovascular Data (Q1–Q10)
3. Reference Questions — COVID-19 Data (Q1–Q11)
4. Turing-Style Practice Questions — Cardio (Q1–Q10)
5. Turing-Style Practice — Joined Cardio + Alco (Q11–Q15)
6. Turing-Style Practice — COVID (Q16–Q20)
7. Additional Potential Questions — Cardio descriptive (Q1–Q10)
8. Additional Potential Questions — Cardio conditional/comparative (Q11–Q18)
9. Additional Potential Questions — Cardio correlation (Q19–Q23)
10. Additional Potential Questions — Cardio data quality (Q24–Q30)
11. Additional Potential Questions — Joins (Q31–Q37)
12. Additional Potential Questions — COVID time-series (Q38–Q47)
13. Additional Potential Questions — COVID data quality (Q48–Q53)

> Run the **Setup** cell first; every later cell depends on the dataframes it creates.


## 1. Setup

Imports, data download, and the preprocessing every later cell depends on.

In [ ]:
import io
import requests
from zipfile import ZipFile

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.optimize import curve_fit

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)


In [ ]:
DATA_FILE = 'https://drive.usercontent.google.com/uc?id=1TMOK9yFDm6nJ2PAC-unBbrG-tR5THJ90&export=download'

resp = requests.get(DATA_FILE)
resp.raise_for_status()
zf = ZipFile(io.BytesIO(resp.content))

contents = {
    name: pd.read_csv(zf.open(name), sep='[,;]', engine='python')
    for name in zf.namelist()
}

assert sorted(contents.keys()) == ['cardio_alco.csv', 'cardio_base.csv', 'covid_data.csv']
print('Loaded:', list(contents.keys()))


In [ ]:
# Cardio: convert age days -> years (rounded down, matching the reference notebook)
cardio = contents['cardio_base.csv'].copy()
cardio['age'] = (cardio['age'] / 365.25).astype(int)

# Drop biologically impossible BP outliers
cardio_clean = cardio[
    cardio['ap_hi'].between(70, 200) & cardio['ap_lo'].between(40, 140)
].copy()

# Alco
alco = contents['cardio_alco.csv'].copy()

# Covid: parse dates
covid = contents['covid_data.csv'].copy()
covid['date'] = pd.to_datetime(covid['date'])

print('cardio:      ', cardio.shape)
print('cardio_clean:', cardio_clean.shape, '(after BP outlier filter)')
print('alco:        ', alco.shape)
print('covid:       ', covid.shape)

# Quick sanity peek
display(cardio.head(3))
display(alco.head(3))
display(covid.head(3))


In [ ]:
# Quick health checks
for name, d in [('cardio', cardio), ('alco', alco), ('covid', covid)]:
    print(f'{name:8s}  shape={d.shape}  dups={d.duplicated().sum():>5}  null_cells={d.isna().sum().sum():>6}')


## 2. Reference Questions — Cardiovascular Data

The actual reference questions for the Turing assessment.

### Cardio Q1 — How much heavier is the age group with the highest average weight than the age group with the lowest weight?

In [ ]:
avg_weight_by_age = cardio_clean.groupby('age')['weight'].mean()
hi_age, hi_w = avg_weight_by_age.idxmax(), avg_weight_by_age.max()
lo_age, lo_w = avg_weight_by_age.idxmin(), avg_weight_by_age.min()
diff = hi_w - lo_w

print(f'Heaviest group:  age {hi_age}, {hi_w:.2f} kg')
print(f'Lightest group:  age {lo_age}, {lo_w:.2f} kg')
print(f'Difference:      {diff:.2f} kg')


### Cardio Q2 — Who smokes more, men or women?

In [ ]:
men_pct   = cardio_clean.loc[cardio_clean['gender'] == 2, 'smoke'].mean() * 100
women_pct = cardio_clean.loc[cardio_clean['gender'] == 1, 'smoke'].mean() * 100

print(f'Men smokers:   {men_pct:.2f}%')
print(f'Women smokers: {women_pct:.2f}%')
print(f'Men are {men_pct / women_pct:.1f}x more likely to smoke.')


### Cardio Q3 — How tall are the tallest 1% of people?

In [ ]:
top_1_threshold = np.percentile(cardio_clean['height'], 99)
print(f'The tallest 1% are taller than {top_1_threshold:.0f} cm.')


### Cardio Q4 — Which two features have the highest Spearman rank correlation?

In [ ]:
spearman = cardio_clean.corr(method='spearman').abs()
np.fill_diagonal(spearman.values, 0)

flat = spearman.unstack().sort_values(ascending=False)
flat = flat[flat.index.get_level_values(0) < flat.index.get_level_values(1)]
print(flat.head(5))


### Cardio Q5 — What percentage of people are more than 2 standard deviations away from the average height?

In [ ]:
mu, sd = cardio_clean['height'].mean(), cardio_clean['height'].std()
mask = (cardio_clean['height'] - mu).abs() > 2 * sd
pct = mask.mean() * 100
print(f'{pct:.2f}% of people are more than 2 sigma from mean height.')


### Cardio Q6 — How many people over 50 years old consume alcohol?

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
n = m[(m['age'] > 50) & (m['alco'] == 1)].shape[0]
print(f'People over 50 who consume alcohol: {n}')


### Cardio Q7 — What percentage of people over 50 years old drink alcohol?

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
over_50 = m[m['age'] > 50]
pct = over_50['alco'].mean() * 100
print(f'{pct:.2f}% of people over 50 drink alcohol.')


### Cardio Q8 — Percentage difference in cholesterol between people over 50 and under 50?

In [ ]:
over  = cardio_clean.loc[cardio_clean['age'] > 50, 'cholesterol'].mean()
under = cardio_clean.loc[cardio_clean['age'] < 50, 'cholesterol'].mean()
pct_diff = (over - under) / under * 100

print(f'Mean cholesterol — over 50:  {over:.4f}')
print(f'Mean cholesterol — under 50: {under:.4f}')
print(f'Percentage difference: {pct_diff:.2f}% higher in over-50s')


### Cardio Q9 — Do smokers or non-smokers have higher cholesterol?

In [ ]:
smk_chol  = cardio_clean.loc[cardio_clean['smoke'] == 1, 'cholesterol']
nsmk_chol = cardio_clean.loc[cardio_clean['smoke'] == 0, 'cholesterol']

print(f'Smoker mean cholesterol:     {smk_chol.mean():.4f}')
print(f'Non-smoker mean cholesterol: {nsmk_chol.mean():.4f}')

u, p = stats.mannwhitneyu(smk_chol, nsmk_chol, alternative='two-sided')
print(f'Mann-Whitney U p-value: {p:.4g}')


### Cardio Q10 — Which statements are true with 95% confidence?

In [ ]:
def claim(name, sample_a, sample_b):
    t, p_two = stats.ttest_ind(sample_a, sample_b, equal_var=False)
    p_one_greater = p_two / 2 if t > 0 else 1 - p_two / 2
    sig = (p_one_greater < 0.05) and (t > 0)
    print(f'{name:55s} t={t:+7.2f}  p_one={p_one_greater:.4g}  -> {"TRUE" if sig else "false"}')

claim('Smokers have higher ap_hi than non-smokers',
      cardio_clean.loc[cardio_clean.smoke==1, 'ap_hi'],
      cardio_clean.loc[cardio_clean.smoke==0, 'ap_hi'])

claim('Smokers have higher cholesterol than non-smokers',
      cardio_clean.loc[cardio_clean.smoke==1, 'cholesterol'],
      cardio_clean.loc[cardio_clean.smoke==0, 'cholesterol'])

claim('Over-50 have higher cholesterol than under-50',
      cardio_clean.loc[cardio_clean.age>50, 'cholesterol'],
      cardio_clean.loc[cardio_clean.age<50, 'cholesterol'])

claim('Men are taller than women',
      cardio_clean.loc[cardio_clean.gender==2, 'height'],
      cardio_clean.loc[cardio_clean.gender==1, 'height'])


## 3. Reference Questions — COVID-19 Data

### COVID Q1 — Which country has the 3rd highest cumulative deaths-per-million?

In [ ]:
latest = covid['date'].max()
snap = covid[(covid['date'] == latest) & covid['continent'].notna()].copy()
snap['deaths_per_m'] = snap['total_deaths'] / snap['population'] * 1e6

ranked = snap.dropna(subset=['deaths_per_m']).nlargest(10, 'deaths_per_m')[
    ['location', 'total_deaths', 'population', 'deaths_per_m']
]
print(ranked.to_string(index=False))
print(f'\n3rd highest: {ranked.iloc[2]["location"]} ({ranked.iloc[2]["deaths_per_m"]:.1f} per million)')


### COVID Q2 — For Italy, days since 2020-02-28 for entries in [2020-02-28, 2020-03-20]

In [ ]:
italy = covid[covid['location'] == 'Italy'].copy()
mask = italy['date'].between('2020-02-28', '2020-03-20')
italy_window = italy[mask].sort_values('date').reset_index(drop=True)
italy_window['days_since_feb28'] = (italy_window['date'] - pd.Timestamp('2020-02-28')).dt.days

print(italy_window[['date', 'days_since_feb28', 'total_cases']].to_string(index=False))


### COVID Q3 / Q11 — Difference between exponential curve and real cases on 2020-03-20

In [ ]:
italy = covid[covid['location'] == 'Italy'].copy().sort_values('date').reset_index(drop=True)
italy = italy[italy['date'] >= '2020-02-28'].reset_index(drop=True)
italy['days'] = (italy['date'] - pd.Timestamp('2020-02-28')).dt.days

train = italy[italy['date'] <= '2020-03-20'].dropna(subset=['total_cases'])

def exp_model(t, a, b):
    return a * np.exp(b * t)

popt, _ = curve_fit(exp_model, train['days'], train['total_cases'], p0=[1, 0.3], maxfev=10000)
a, b = popt
print(f'Fit: total_cases ~= {a:.2f} * exp({b:.4f} * days)')

target_days = (pd.Timestamp('2020-03-20') - pd.Timestamp('2020-02-28')).days
predicted = exp_model(target_days, a, b)
actual = italy.loc[italy['date'] == '2020-03-20', 'total_cases'].iloc[0]

print(f'Predicted on 2020-03-20: {predicted:,.0f}')
print(f'Actual on 2020-03-20:    {actual:,.0f}')
print(f'Difference (pred-actual): {predicted - actual:,.0f}')

# Visual check
plt.figure(figsize=(10, 5))
plt.plot(italy['date'], italy['total_cases'], 'o-', label='Actual')
days_grid = np.arange(0, italy['days'].max() + 1)
plt.plot(pd.Timestamp('2020-02-28') + pd.to_timedelta(days_grid, 'D'),
         exp_model(days_grid, a, b), '--', label='Exponential fit')
plt.legend(); plt.title('Italy COVID — actual vs exponential fit')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


### COVID Q4 — Same as Cardio Q1 (heaviest vs lightest age group)
See Cardio Q1 above.

### COVID Q5 — Do people over 50 have higher cholesterol than the rest?

In [ ]:
over  = cardio_clean.loc[cardio_clean['age'] > 50, 'cholesterol']
rest  = cardio_clean.loc[cardio_clean['age'] <= 50, 'cholesterol']

print(f'Over 50 mean:  {over.mean():.4f}')
print(f'<= 50 mean:    {rest.mean():.4f}')

u, p = stats.mannwhitneyu(over, rest, alternative='greater')
print(f'Mann-Whitney p (one-sided over > rest): {p:.4g}')
print('YES' if p < 0.05 else 'NO')


### COVID Q6 — Same as Cardio Q2 (men vs women smoking)

### COVID Q7 — Same as Cardio Q3 (tallest 1%)

### COVID Q8 — Same as Cardio Q4 (highest Spearman correlation)

### COVID Q9 — Same as Cardio Q5 (>2 SD from mean height)

### COVID Q10 — When did Italy vs Germany case difference first exceed 10,000?

In [ ]:
italy = covid.loc[covid['location'] == 'Italy', ['date', 'total_cases']].rename(
    columns={'total_cases': 'italy'})
germany = covid.loc[covid['location'] == 'Germany', ['date', 'total_cases']].rename(
    columns={'total_cases': 'germany'})

merged = italy.merge(germany, on='date').sort_values('date')
merged['diff'] = (merged['italy'] - merged['germany']).abs()

first_over = merged[merged['diff'] > 10_000].head(1)
print(first_over[['date', 'italy', 'germany', 'diff']].to_string(index=False))


## 4. Turing-Style Practice Questions — Cardio

### Q1 — Heaviest vs lightest age group (same as reference Cardio Q1)
See section 2.

### Q2 — Men vs women smoking (same as reference Cardio Q2)

### Q3 — Tallest 1% (same as reference Cardio Q3)

### Q4 — Absolute difference in mean systolic BP between >50 and <=50 age groups

In [ ]:
above = cardio_clean.loc[cardio_clean['age'] > 50, 'ap_hi'].mean()
below = cardio_clean.loc[cardio_clean['age'] <= 50, 'ap_hi'].mean()
print(f'>50 mean ap_hi:  {above:.2f}')
print(f'<=50 mean ap_hi: {below:.2f}')
print(f'Difference:      {above - below:.2f} mmHg')


### Q5 — Which gender has higher mean cholesterol, and by what margin?

In [ ]:
g_chol = cardio_clean.groupby('gender')['cholesterol'].mean()
print(g_chol)
print(f'Difference: {abs(g_chol[2] - g_chol[1]):.4f}')


### Q6 — % of dataset that is overweight (BMI > 25)

In [ ]:
bmi = cardio_clean['weight'] / (cardio_clean['height'] / 100) ** 2
pct = (bmi > 25).mean() * 100
print(f'{pct:.2f}% of the dataset is overweight (BMI > 25).')


### Q7 — Among hypertensives (ap_hi > 130 OR ap_lo > 90), what % are smokers?

In [ ]:
hyper = cardio_clean[(cardio_clean['ap_hi'] > 130) | (cardio_clean['ap_lo'] > 90)]
pct = hyper['smoke'].mean() * 100
print(f'Smoker rate among hypertensives: {pct:.2f}%')


### Q8 — Which single age has the most hypertensive patients?

In [ ]:
hyper = cardio_clean[(cardio_clean['ap_hi'] > 130) | (cardio_clean['ap_lo'] > 90)]
print(hyper['age'].value_counts().head(5))
top_age = hyper['age'].value_counts().idxmax()
print(f'Top age: {top_age} with {hyper["age"].value_counts().max()} patients')


### Q9 — Pearson correlation between weight and ap_hi

In [ ]:
r = cardio_clean['weight'].corr(cardio_clean['ap_hi'])
print(f'Pearson r(weight, ap_hi) = {r:.4f}')


### Q10 — Strongest absolute correlation pair (Pearson)

In [ ]:
corr = cardio_clean.corr().abs()
np.fill_diagonal(corr.values, 0)
flat = corr.unstack().sort_values(ascending=False)
flat = flat[flat.index.get_level_values(0) < flat.index.get_level_values(1)]
print(flat.head(5))


## 5. Turing-Style Practice — Joined Cardio + Alco

### Q11 — Inner-join match count

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
print(f'Joined rows: {len(m)}')
print(f'IDs in alco missing from cardio: {(~alco["id"].isin(cardio_clean["id"])).sum()}')


### Q12 — % of people over 50 who consume alcohol (same as ref Cardio Q7)

### Q13 — Which age (single integer) consumes alcohol the most?

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
drinkers = m[m['alco'] == 1]
print(drinkers['age'].value_counts().head(5))


### Q14 — Are alcohol consumers more likely to have hypertension? (chi-square)

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
m['hyper'] = ((m['ap_hi'] > 130) | (m['ap_lo'] > 90)).astype(int)

drinker_rate    = m.loc[m['alco'] == 1, 'hyper'].mean()
nondrinker_rate = m.loc[m['alco'] == 0, 'hyper'].mean()

table = pd.crosstab(m['alco'], m['hyper'])
chi2, p, _, _ = stats.chi2_contingency(table)

print(f'Hypertension rate, drinkers:    {drinker_rate*100:.2f}%')
print(f'Hypertension rate, nondrinkers: {nondrinker_rate*100:.2f}%')
print(f'Chi-square p-value: {p:.4g}')
print('Significant at 95%' if p < 0.05 else 'Not significant')


### Q15 — Smokers vs non-smokers — higher ap_hi at 95% confidence?

In [ ]:
smk = cardio_clean.loc[cardio_clean['smoke'] == 1, 'ap_hi']
non = cardio_clean.loc[cardio_clean['smoke'] == 0, 'ap_hi']
t, p = stats.ttest_ind(smk, non, equal_var=False)
print(f'Smoker mean: {smk.mean():.2f}, non-smoker mean: {non.mean():.2f}')
print(f't = {t:.3f}, two-sided p = {p:.4g}')
print('Smokers significantly higher' if (p < 0.05 and t > 0) else 'Not significantly higher')


## 6. Turing-Style Practice — COVID

### Q16 — Date of global peak in daily new cases

In [ ]:
world = covid[covid['location'] == 'World']
if not world.empty:
    peak_date = world.loc[world['new_cases'].idxmax(), 'date']
    peak_val  = world['new_cases'].max()
    print(f'Global peak (World row): {peak_date.date()} with {peak_val:,.0f} new cases')
else:
    by_date = covid.groupby('date')['new_cases'].sum()
    print('Global peak (summed):', by_date.idxmax().date(), f'{by_date.max():,.0f}')


### Q17 — Country (pop > 10M) with highest deaths-per-million on latest date

In [ ]:
latest = covid['date'].max()
snap = covid[(covid['date'] == latest) & covid['continent'].notna()]
snap = snap[snap['population'] > 10_000_000].copy()
snap['deaths_per_m'] = snap['total_deaths'] / snap['population'] * 1e6
print(snap.nlargest(5, 'deaths_per_m')[['location', 'total_deaths', 'population', 'deaths_per_m']])


### Q18 — 7-day rolling avg of Italy's new cases on 2020-04-15

In [ ]:
italy = covid[covid['location'] == 'Italy'].sort_values('date').copy()
italy['ma7'] = italy['new_cases'].rolling(7).mean()
val = italy.loc[italy['date'] == '2020-04-15', 'ma7']
print(f'Italy 7-day MA on 2020-04-15:', float(val.iloc[0]) if len(val) else 'no data')


### Q19 — Median days from first case to 1,000th case across countries

In [ ]:
def days_to_1000(g):
    g = g.sort_values('date')
    first = g.loc[g['total_cases'] > 0, 'date'].min()
    thou  = g.loc[g['total_cases'] >= 1000, 'date'].min()
    if pd.isna(first) or pd.isna(thou):
        return np.nan
    return (thou - first).days

per_country = covid[covid['continent'].notna()].groupby('location').apply(days_to_1000)
print(f'Median days first->1000: {per_country.median():.0f}')
print(f'Countries that never reached 1000: {per_country.isna().sum()}')


### Q20 — Continent with highest case-fatality rate on latest date

In [ ]:
latest = covid['date'].max()
snap = covid[(covid['date'] == latest) & covid['continent'].notna()]
g = snap.groupby('continent')[['total_cases', 'total_deaths']].sum()
g['cfr'] = g['total_deaths'] / g['total_cases']
print(g.sort_values('cfr', ascending=False))


## 7. Additional Potential Questions — Cardio Descriptive

### P1 — Average weight of people aged 50+

In [ ]:
print(cardio.loc[cardio['age'] >= 50, 'weight'].mean())

### P2 — Median height across the full dataset

In [ ]:
print(cardio['height'].median())

### P3 — % with cholesterol > 1

In [ ]:
print((cardio['cholesterol'] > 1).mean() * 100, '%')

### P4 — Number of smokers

In [ ]:
print((cardio['smoke'] == 1).sum())

### P5 — Std deviation of ap_hi (raw vs cleaned)

In [ ]:
print('Raw   :', cardio['ap_hi'].std())
print('Clean :', cardio_clean['ap_hi'].std())


### P6 — Average BMI

In [ ]:
bmi = cardio['weight'] / (cardio['height'] / 100) ** 2
print('Mean BMI:', bmi.mean())


### P7 — Mean weight by gender

In [ ]:
print(cardio.groupby('gender')['weight'].mean())

### P8 — % with height > mean + 1 std

In [ ]:
thr = cardio['height'].mean() + cardio['height'].std()
print((cardio['height'] > thr).mean() * 100, '%')


### P9 — 90th percentile of weight

In [ ]:
print(cardio['weight'].quantile(0.90))

### P10 — Distinct ages in years

In [ ]:
print(cardio['age'].nunique())

## 8. Additional Potential — Cardio Conditional / Comparative

### P11 — Of people with ap_hi > avg, % over 50?

In [ ]:
hi = cardio_clean[cardio_clean['ap_hi'] > cardio_clean['ap_hi'].mean()]
print((hi['age'] > 50).mean() * 100, '%')


### P12 — Men vs women smoking — absolute and rate

In [ ]:
print(cardio_clean.groupby('gender')['smoke'].agg(['sum', 'mean']))

### P13 — Avg cholesterol — smokers vs non-smokers

In [ ]:
print(cardio_clean.groupby('smoke')['cholesterol'].mean())

### P14 — Avg weight — cardio vs no-cardio (if column exists)

In [ ]:
if 'cardio' in cardio_clean.columns:
    print(cardio_clean.groupby('cardio')['weight'].mean())
else:
    print('No cardio column in this version of the dataset.')


### P15 — Above vs below median age — higher cholesterol?

In [ ]:
m = cardio_clean['age'].median()
print(cardio_clean.assign(older=cardio_clean['age'] > m).groupby('older')['cholesterol'].mean())


### P16 — Of people > 55 yrs, % with cholesterol == 3?

In [ ]:
older = cardio_clean[cardio_clean['age'] > 55]
print((older['cholesterol'] == 3).mean() * 100, '%')


### P17 — Of people with ap_hi >= 140, fraction smokers?

In [ ]:
hyper = cardio_clean[cardio_clean['ap_hi'] >= 140]
print((hyper['smoke'] == 1).mean())


### P18 — P(cardio = 1 | age > 60), if column exists

In [ ]:
if 'cardio' in cardio_clean.columns:
    print(cardio_clean[cardio_clean['age'] > 60]['cardio'].mean())
else:
    print('No cardio column in this version of the dataset.')


## 9. Additional Potential — Correlation

### P19 — Strongest correlate of cardio (Pearson)

In [ ]:
if 'cardio' in cardio_clean.columns:
    cols = ['age', 'weight', 'cholesterol', 'ap_hi', 'cardio']
    print(cardio_clean[cols].corr()['cardio'].abs().sort_values(ascending=False))
else:
    print('No cardio column.')


### P20 — Pearson r(height, weight)

In [ ]:
print(cardio_clean['height'].corr(cardio_clean['weight']))

### P21 — Age vs systolic vs diastolic

In [ ]:
print('age vs ap_hi:', cardio_clean['age'].corr(cardio_clean['ap_hi']))
print('age vs ap_lo:', cardio_clean['age'].corr(cardio_clean['ap_lo']))


### P22 — Strongest negative correlation among numerics

In [ ]:
c = cardio_clean.select_dtypes('number').corr()
flat = c.unstack().sort_values()
flat = flat[flat.index.get_level_values(0) < flat.index.get_level_values(1)]
print(flat.head(5))


### P23 — Does corr(weight, ap_hi) change after outlier removal?

In [ ]:
print('Raw   :', cardio['weight'].corr(cardio['ap_hi']))
print('Clean :', cardio_clean['weight'].corr(cardio_clean['ap_hi']))


## 10. Additional Potential — Cardio Data Quality

### P24 — Mean age in years

In [ ]:
print(cardio['age'].mean())

### P25 — Rows with ap_lo > ap_hi

In [ ]:
print((cardio['ap_lo'] > cardio['ap_hi']).sum())

### P26 — Rows with height outside 100-220 cm

In [ ]:
print((~cardio['height'].between(100, 220)).sum())

### P27 — Rows with weight outside 30-200 kg

In [ ]:
print((~cardio['weight'].between(30, 200)).sum())

### P28 — Duplicate id values

In [ ]:
print(cardio['id'].duplicated().sum())

### P29 — Mean ap_hi after cleaning

In [ ]:
print(cardio_clean['ap_hi'].mean())

### P30 — % excluded by trimming top/bottom 1% on ap_hi and ap_lo

In [ ]:
lo_h, hi_h = cardio['ap_hi'].quantile([0.01, 0.99])
lo_l, hi_l = cardio['ap_lo'].quantile([0.01, 0.99])
mask = cardio['ap_hi'].between(lo_h, hi_h) & cardio['ap_lo'].between(lo_l, hi_l)
print((1 - mask.mean()) * 100, '%')


## 11. Additional Potential — Joins

### P31 — % of alcohol consumers with cardiovascular disease

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
if 'cardio' in m.columns:
    drinkers = m[m['alco'] == 1]
    print(drinkers['cardio'].mean() * 100, '%')
else:
    print('No cardio column.')


### P32 — Inner-join match count

In [ ]:
print(cardio_clean.merge(alco, on='id', how='inner').shape[0])

### P33 — IDs in alco not in cardio_base

In [ ]:
print('Missing:', (~alco['id'].isin(cardio_clean['id'])).sum())

### P34 — Avg age of alcohol consumers

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
print(m.loc[m['alco'] == 1, 'age'].mean())


### P35 — Cardio rate: drinkers vs non-drinkers

In [ ]:
m = cardio_clean.merge(alco, on='id', how='inner')
if 'cardio' in m.columns:
    print(m.groupby('alco')['cardio'].mean())
else:
    print('No cardio column.')


### P36 — Diagnose duplicate-ID inflation in alco

In [ ]:
print(alco['id'].value_counts().head())

### P37 — Compare join strategies

In [ ]:
inner = cardio_clean.merge(alco, on='id', how='inner').shape[0]
left  = cardio_clean.merge(alco, on='id', how='left').shape[0]
outer = cardio_clean.merge(alco, on='id', how='outer').shape[0]
print(f'inner={inner}, left={left}, outer={outer}')


## 12. Additional Potential — COVID Time-Series

### P38 — Date of global peak (already in section 6)

### P39 — Country with highest total deaths on latest date

In [ ]:
latest = covid['date'].max()
snap = covid[(covid['date'] == latest) & covid['continent'].notna()]
print(snap.nlargest(1, 'total_deaths')[['location', 'total_deaths']])


### P40 — Highest deaths-per-million among countries with pop > 1M

In [ ]:
latest = covid['date'].max()
snap = covid[(covid['date'] == latest) & covid['continent'].notna()].copy()
snap['dpm'] = snap['total_deaths'] / snap['population'] * 1e6
print(snap[snap['population'] > 1_000_000].nlargest(1, 'dpm')[['location', 'dpm']])


### P41 — 7-day MA new cases for a country on a date (parameterise)

In [ ]:
def ma7(country, date):
    c = covid[covid['location'] == country].sort_values('date').copy()
    c['ma7'] = c['new_cases'].rolling(7).mean()
    val = c.loc[c['date'] == date, 'ma7']
    return float(val.iloc[0]) if len(val) else None

print('Italy 2020-04-15:', ma7('Italy', '2020-04-15'))
print('Germany 2020-04-15:', ma7('Germany', '2020-04-15'))


### P42 — Continent with highest case-fatality rate (already in section 6)

### P43 — Countries with zero cases on first date

In [ ]:
first = covid['date'].min()
print((covid[(covid['date'] == first) & covid['continent'].notna()]['total_cases'].fillna(0) == 0).sum())


### P44 — Doubling time for a country between two dates

In [ ]:
def doubling(country, d1, d2):
    c = covid[covid['location'] == country].set_index('date')
    C1, C2 = c.loc[d1, 'total_cases'], c.loc[d2, 'total_cases']
    days = (pd.Timestamp(d2) - pd.Timestamp(d1)).days
    return days * np.log(2) / np.log(C2 / C1)

print('Italy doubling 2020-03-01 -> 2020-03-15:', doubling('Italy', '2020-03-01', '2020-03-15'))


### P45 — Top 5 countries by total cases per capita (latest)

In [ ]:
latest = covid['date'].max()
snap = covid[(covid['date'] == latest) & covid['continent'].notna()].copy()
snap['cases_pc'] = snap['total_cases'] / snap['population']
print(snap.nlargest(5, 'cases_pc')[['location', 'cases_pc']])


### P46 — First date country X had > 1000 daily new cases (parameterise)

In [ ]:
def first_over_1k(country):
    c = covid[(covid['location'] == country) & (covid['new_cases'] > 1000)].sort_values('date')
    return c['date'].min()

print('Italy:', first_over_1k('Italy'))
print('Germany:', first_over_1k('Germany'))


### P47 — Avg days between first reported case and 100th case across countries

In [ ]:
def gap(g):
    g = g.sort_values('date')
    first = g.loc[g['total_cases'] > 0, 'date'].min()
    hundred = g.loc[g['total_cases'] >= 100, 'date'].min()
    return (hundred - first).days if pd.notna(first) and pd.notna(hundred) else None

gaps = covid[covid['continent'].notna()].groupby('location').apply(gap)
print('Mean days first -> 100:', gaps.dropna().mean())


## 13. Additional Potential — COVID Data Quality

### P48 — Cumulative or daily — spot non-monotonic countries

In [ ]:
chk = (covid.sort_values(['location','date'])
              .groupby('location')['total_cases']
              .apply(lambda s: (s.diff().fillna(0) < 0).sum()))
print(chk[chk > 0].head(10))


### P49 — Countries with new_cases < 0

In [ ]:
neg = covid[covid['new_cases'] < 0]
print('Distinct countries with negative new_cases:', neg['location'].nunique())
print(neg['location'].value_counts().head())


### P50 — Missing (country, date) pairs

In [ ]:
# Skip if dataset is too large; this can be expensive
locs = covid['location'].unique()
dates = pd.date_range(covid['date'].min(), covid['date'].max())
expected = pd.MultiIndex.from_product([locs, dates], names=['location', 'date'])
present = covid.set_index(['location', 'date']).index
missing = expected.difference(present)
print(f'Missing (country, date) pairs: {len(missing):,}')


### P51 — Continent rollups vs sum of member countries

In [ ]:
# Sum from country rows (where continent is set)
sum_from_countries = (covid[covid['continent'].notna()]
                       .groupby(['continent', 'date'])['total_cases']
                       .sum()
                       .reset_index()
                       .rename(columns={'total_cases': 'sum_from_countries'}))

# Official continent rollup rows (where continent is NaN, location is a continent name)
continents = ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']
official = (covid[covid['location'].isin(continents)]
              [['location', 'date', 'total_cases']]
              .rename(columns={'location': 'continent', 'total_cases': 'official'}))

cmp = sum_from_countries.merge(official, on=['continent', 'date'], how='outer')
cmp['delta'] = cmp['official'] - cmp['sum_from_countries']
print(cmp.dropna().sort_values('delta', key=abs, ascending=False).head(10))


### P52 — Why total_cases sometimes decreases day-over-day

**Three causes:**
1. Retroactive correction of prior over-counts.
2. Re-classification (probable -> confirmed flipped back).
3. Pipeline / source change.
A decrease in a *cumulative* column is a red flag, not a feature.

### P53 — Two dashboards disagreeing on country totals — three reasons

1. Different snapshot cut-off times (00:00 UTC vs 06:00 UTC).
2. Different territory inclusion rules (France with vs without overseas territories).
3. Different definitions (confirmed only vs confirmed + probable; including reinfections or not).

## Done

- All cardio questions use `cardio_clean` (BP outliers dropped). For raw versions, swap to `cardio`.
- All cardio age comparisons assume **age in years** — already converted in setup.
- All COVID country queries filter `continent.notna()` to drop World/continent rollup rows.
- For SQL / event-data / metric-definition concept questions (Q54–Q67 in `potential_questions.md`),
  see `answers.md` — they don't run as Python so they're omitted here.
